<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230824_ar2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler


In [23]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')


In [52]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 34

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)




Epoch 1/50
157/157 [==============================] - 29s 115ms/step - loss: 9.1665e-04 - mae: 0.0204 - val_loss: 1.5362e-04 - val_mae: 0.0098
Epoch 2/50
157/157 [==============================] - 16s 102ms/step - loss: 1.8529e-04 - mae: 0.0105 - val_loss: 1.8655e-04 - val_mae: 0.0108
Epoch 3/50
157/157 [==============================] - 17s 105ms/step - loss: 1.3051e-04 - mae: 0.0087 - val_loss: 1.7526e-04 - val_mae: 0.0104
Epoch 4/50
157/157 [==============================] - 16s 105ms/step - loss: 1.0656e-04 - mae: 0.0079 - val_loss: 2.1032e-04 - val_mae: 0.0114
Epoch 5/50
157/157 [==============================] - 16s 105ms/step - loss: 9.7107e-05 - mae: 0.0075 - val_loss: 1.8976e-04 - val_mae: 0.0108
Epoch 6/50
157/157 [==============================] - 18s 114ms/step - loss: 8.4575e-05 - mae: 0.0070 - val_loss: 1.6492e-04 - val_mae: 0.0101
Epoch 7/50
157/157 [==============================] - 17s 105ms/step - loss: 7.6615e-05 - mae: 0.0067 - val_loss: 1.9436e-04 - val_mae: 0.0110

In [53]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 34

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)




Epoch 1/50
157/157 [==============================] - 29s 115ms/step - loss: 0.0011 - mae: 0.0231 - val_loss: 0.0012 - val_mae: 0.0268
Epoch 2/50
157/157 [==============================] - 17s 106ms/step - loss: 2.9477e-04 - mae: 0.0130 - val_loss: 9.3462e-04 - val_mae: 0.0240
Epoch 3/50
157/157 [==============================] - 17s 106ms/step - loss: 2.1884e-04 - mae: 0.0112 - val_loss: 9.0312e-04 - val_mae: 0.0236
Epoch 4/50
157/157 [==============================] - 18s 117ms/step - loss: 1.8436e-04 - mae: 0.0101 - val_loss: 0.0011 - val_mae: 0.0259
Epoch 5/50
157/157 [==============================] - 17s 108ms/step - loss: 1.5548e-04 - mae: 0.0092 - val_loss: 0.0013 - val_mae: 0.0278
Epoch 6/50
157/157 [==============================] - 17s 106ms/step - loss: 1.4042e-04 - mae: 0.0088 - val_loss: 0.0012 - val_mae: 0.0266
Epoch 7/50
157/157 [==============================] - 17s 106ms/step - loss: 1.2839e-04 - mae: 0.0084 - val_loss: 0.0012 - val_mae: 0.0270
Epoch 8/50
157/157 [===

In [27]:
predictions_30s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [30]:
predictions_40s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [33]:
predictions_50s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [36]:
predictions_70s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [39]:
predictions_100s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [42]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [45]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [48]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [51]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [54]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [55]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.concat([predictions_30c,predictions_30c2],axis=1)
c40 = pd.concat([predictions_40c,predictions_40c2],axis=1)
c50 = pd.concat([predictions_50c,predictions_50c2],axis=1)
c70 = pd.concat([predictions_70c,predictions_70c2],axis=1)
c100 = pd.concat([predictions_100c,predictions_100c2],axis=1)

s30 = pd.concat([predictions_30s,predictions_30s2],axis=1)
s40 = pd.concat([predictions_40s,predictions_40s2],axis=1)
s50 = pd.concat([predictions_50s,predictions_50s2],axis=1)
s70 = pd.concat([predictions_70s,predictions_70s2],axis=1)
s100 = pd.concat([predictions_100s,predictions_100s2],axis=1)

In [56]:
c30  = pd.DataFrame(c30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c40  = pd.DataFrame(c40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c50  = pd.DataFrame(c50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c70  = pd.DataFrame(c70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c100  = pd.DataFrame(c100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s30  = pd.DataFrame(s30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s40  = pd.DataFrame(s40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s50  = pd.DataFrame(s50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s70  = pd.DataFrame(s70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s100  = pd.DataFrame(s100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])

In [57]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

s30.columns = ['YL_M1_B1_W1_s30', 'YR_M1_B1_W1_s30', 'YL_M1_B1_W2_s30', 'YR_M1_B1_W2_s30']
s40.columns = ['YL_M1_B1_W1_s40', 'YR_M1_B1_W1_s40', 'YL_M1_B1_W2_s40', 'YR_M1_B1_W2_s40']
s50.columns = ['YL_M1_B1_W1_s50', 'YR_M1_B1_W1_s50', 'YL_M1_B1_W2_s50', 'YR_M1_B1_W2_s50']
s70.columns = ['YL_M1_B1_W1_s70', 'YR_M1_B1_W1_s70', 'YL_M1_B1_W2_s70', 'YR_M1_B1_W2_s70']
s100.columns = ['YL_M1_B1_W1_s100', 'YR_M1_B1_W1_s100', 'YL_M1_B1_W2_s100', 'YR_M1_B1_W2_s100']

In [58]:
c30

,YL_M1_B1_W1_c30,YR_M1_B1_W1_c30,YL_M1_B1_W2_c30,YR_M1_B1_W2_c30
0,0.011262,-0.005649,0.036356,-0.029158
1,0.007861,-0.000626,0.042536,-0.035456
2,0.001282,0.006832,0.046254,-0.039536
3,-0.003255,0.010997,0.045543,-0.039578
4,-0.003809,0.010346,0.043032,-0.037476
...,...,...,...,...
1994,0.001173,0.002006,0.040964,-0.032101
1995,0.001593,0.000869,0.033873,-0.026747
1996,-0.000765,0.004627,0.017349,-0.012421
1997,0.000741,0.004405,-0.001862,0.006689


In [59]:
answer = pd.concat([answer_sample.Distance, s30,s40,s50,s70,s100,c30,c40,c50,c70,c100], axis=1)

In [60]:
answer

,Distance,YL_M1_B1_W1_s30,YR_M1_B1_W1_s30,YL_M1_B1_W2_s30,YR_M1_B1_W2_s30,YL_M1_B1_W1_s40,YR_M1_B1_W1_s40,YL_M1_B1_W2_s40,YR_M1_B1_W2_s40,YL_M1_B1_W1_s50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.028384,-0.009307,0.050305,-0.039862,0.001097,0.004781,-0.008788,0.011626,0.005095,...,-0.004719,0.010119,0.005087,0.005394,-0.006023,0.012799,0.004814,0.003392,-0.002595,0.008437
1,2500.50,0.024338,-0.004145,0.057247,-0.046711,-0.004025,0.009684,-0.003571,0.006405,0.000881,...,0.004126,0.000233,0.000478,0.011997,0.002087,0.004420,-0.000278,0.009597,0.005931,-0.001176
2,2500.75,0.012553,0.006805,0.063194,-0.053155,-0.013290,0.018959,0.001678,0.001484,-0.009774,...,0.014581,-0.011587,-0.011076,0.024911,0.011973,-0.005742,-0.012386,0.022483,0.014990,-0.011470
3,2501.00,0.003109,0.014914,0.064723,-0.055645,-0.022206,0.027796,0.004435,-0.001012,-0.020869,...,0.023001,-0.021077,-0.022814,0.037006,0.019560,-0.013544,-0.024799,0.034721,0.021476,-0.019013
4,2501.25,0.000435,0.016213,0.062399,-0.053904,-0.025385,0.031423,0.004928,-0.001319,-0.024554,...,0.028132,-0.026668,-0.026804,0.041033,0.023433,-0.017569,-0.029550,0.039225,0.024714,-0.022833
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,0.007045,-0.003057,0.046700,-0.039268,0.009411,-0.004372,0.039906,-0.030757,0.013255,...,0.032182,-0.026118,0.011846,-0.006649,0.039586,-0.028408,0.006889,-0.003217,0.034842,-0.027046
1995,2999.00,0.006493,-0.002899,0.039526,-0.032418,0.009174,-0.004983,0.035504,-0.026374,0.010106,...,0.029943,-0.023143,0.011875,-0.006726,0.033119,-0.023147,0.005771,-0.001188,0.033454,-0.025300
1996,2999.25,0.003142,0.000100,0.024227,-0.017008,0.005191,-0.001118,0.021572,-0.013539,0.005020,...,0.020190,-0.013910,0.007194,-0.001568,0.019123,-0.011604,0.000512,0.005465,0.022129,-0.014120
1997,2999.50,0.007180,-0.003903,0.007348,0.000119,0.007730,-0.003656,0.004885,0.001992,0.007374,...,0.003879,0.000581,0.008337,-0.002615,0.003775,0.001761,0.000407,0.005818,0.005046,0.001854


In [61]:
answer.to_csv('/content/drive/My Drive/철도/answer20230824_ar2.csv', index=False)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 예측과 실제값 간의 MAE, MSE, RMSE, R-squared 계산
mae = mean_absolute_error(y_test_s, predictions_s)
mse = mean_squared_error(y_test_s, predictions_s)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_s, predictions_s)

# 결과 출력
print("Evaluation metrics for data_s:")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)


Evaluation metrics for data_s:
MAE: 0.004689485690516887
MSE: 5.018317484975397e-05
RMSE: 0.007084008388599915
R-squared: 0.9032700239745032
